In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
import pandas as pd
import numpy as np
import anndata as ad
import math
import random
import hyperopt
import time

from hyperopt import fmin, tpe, hp, STATUS_OK, STATUS_FAIL, Trials, space_eval
from functools import partial
from hyperopt import hp

In [9]:
df = {}

def create_config(params):
    # params['n_layers_decoders'] = params['n_layers_encoders']
    df[len(df)] = params
    
    #with open('/lustre/groups/ml01/workspace/anastasia.litinetskaya/configs/haniffa/test.json') as f:
     #   d = json.load(f)
    
    #counter = len(p) + 63
    
#     d['kl'] = params['kl']
#     d['model']['train']['lr'] = params['lr']
#     d['model']['query_train']['lr'] = params['lr']
#     d['model']['model_params']['scoring'] = params["scoring"]
#     d['model']['model_params']['attn_dim'] = params["attn_dim"]
#     d['model']['model_params']['drop_attn'] = params["drop_attn"]
#     d['model']['train']['batch_size'] = params["batch_size"]
#     d['model']['query_train']['batch_size'] = params["batch_size"]
#     d['model']['model_params']['patient_batch_size'] = int(params["batch_size"] // 2)
    
#     if params['covs'] == 'both':
#         d['model']['setup_params']['categorical_covariate_keys'] = ["Site", "patient_id"]
#     elif params['covs'] == "patient_id":
#         d['model']['setup_params']['categorical_covariate_keys'] = ["patient_id"]
    
#     d['model']['model_params']['n_layers_encoders'] = [params['depth']] * 2
#     d['model']['model_params']['n_layers_decoders'] = [params['depth']] * 2
#     d['model']['model_params']['n_layers_classifier'] = params['depth']
    
    
#     with open(f'/lustre/groups/ml01/workspace/anastasia.litinetskaya/configs/haniffa/{counter}.json', 'w') as f:
#         json.dump(d, f, indent=4)
    
    return -1


def objective(params, trials, space):

    if len(trials.trials) > 1:
        for x in trials.trials[:-1]:
            space_point_index = dict(
                [
                    (key, value[0])
                    for key, value in x["misc"]["vals"].items()
                    if len(value) > 0
                ]
            )
            if params == space_eval(space, space_point_index):
                # optimization algorithm
                loss = x["result"]["loss"]
                return {"loss": loss, "status": STATUS_FAIL, "eval_time": time.time()}
            if params['batch_size'] == 256 and params['patient_batch_size'] == 256:
                return {"loss": -1, "status": STATUS_FAIL, "eval_time": time.time()}

    df = {}
    loss = create_config(params)
#    df = pd.DataFrame(df)
 #   df.to_csv('params.tsv', sep='\t')

    return {"loss": loss, "status": STATUS_OK, "eval_time": time.time()}


def hyper_optimize(max_evals):
    # define the search space
    space = hp.choice(
        "params",
        [
            {
#                 "kl": hp.choice("kl", {1e-4, 1e-3, 1e-2, 1e-5}),
#                 "lr": hp.choice("lr", {1e-5, 1e-4, 1e-3}),
#                 #"covs": hp.choice("covs", {"both"}),
#                 "scoring": hp.choice("scoring", {"gated_attn", "attn"}),
#                 "batch_size": hp.choice("batch_size", {256, 512}),
#                 "drop_attn": hp.choice("drop_attn", {True, False}),
#                 "attention_dropout": hp.choice("attention_dropout", {True, False}),
#                 "attn_dim": hp.choice("attn_dim", {16, 32}),
#                 "z_dim": hp.choice("z_dim", {16, 32}),
#                 "cond_dim": hp.choice("cond_dim", {8, 16, 32}),
#                 "n_layers_encoders": hp.choice("n_layers_encoders", {1, 2, 3}),
#                 "n_layers_classifier": hp.choice("n_layers_classifier", {1, 2, 3}),
#                 "patient_batch_size": hp.choice("patient_batch_size", {128, 256}),
#                 "initialization": hp.choice("initialization", {None, "kaiming"}),
                "kl": hp.choice("kl", {1e-4, 1e-3, 1e-5, 1e-2}),
                "lr": hp.choice("lr", {1e-5, 5e-5}),
                #"covs": hp.choice("covs", {"both"}),
                #"scoring": hp.choice("scoring", {"gated_attn"}),
                "batch_size": hp.choice("batch_size", {256, 512}),
                "drop_attn": hp.choice("drop_attn", {True, False}),
                "attention_dropout": hp.choice("attention_dropout", {False, True}),
                "attn_dim": hp.choice("attn_dim", {8, 16, 32}),
                "z_dim": hp.choice("z_dim", {8, 16, 32}),
                "cond_dim": hp.choice("cond_dim", {8, 16, 32}),
               # "n_layers_encoders": hp.choice("n_layers_encoders", {2}),
                "n_layers_classifier": hp.choice("n_layers_classifier", {1, 2, 3}),
                "patient_batch_size": hp.choice("patient_batch_size", {128, 256}),
#                "initialization": hp.choice("initialization", {"kaiming"}),
                "class_loss_coef": hp.choice("class_loss_coef", {1.0, 10, 100}),
            }
        ]
    )

    trials = Trials()
    fmin_objective = partial(objective, trials=trials, space=space)
    best = fmin(
        fmin_objective,
        space=space,
        algo=tpe.suggest,
        max_evals=max_evals,
        trials=trials,
    )
    return best

In [10]:
best = hyper_optimize(100)
print(best)

100%|██████████| 100/100 [00:02<00:00, 47.53trial/s, best loss: -1.0]
{'attention_dropout': 0, 'attn_dim': 1, 'batch_size': 0, 'class_loss_coef': 2, 'cond_dim': 0, 'drop_attn': 0, 'kl': 2, 'lr': 1, 'n_layers_classifier': 1, 'params': 0, 'patient_batch_size': 0, 'z_dim': 2}


In [11]:
len(df.keys())

53

In [12]:
pd.DataFrame(df).T.to_csv('/lustre/groups/ml01/projects/2022_multigrate_anastasia.litinetskaya/snakemake/hyperparam/data/params.tsv', sep='\t')